# 正则化、Dropout 与 Weight Decay

## 本节目标

模型过强而数据较少时容易过拟合。本节通过小数据分类任务观察训练集和测试集表现，并引入 Dropout 与 weight decay。

完成本节后，你应该能够：

- 说清本节主题解决了什么问题。
- 读懂核心 PyTorch API 的输入输出。
- 运行最小代码示例并解释结果。
- 修改实验参数并观察变化。

## 背景问题

本节关注的问题是：模型过强而数据较少时容易过拟合。本节通过小数据分类任务观察训练集和测试集表现，并引入 Dropout 与 weight decay。

学习时建议先运行代码，再回头看解释。对初学者来说，能观察 shape、loss、accuracy 或可视化结果，比只记概念更重要。

## 核心概念

- 输入和输出的 shape 必须明确。
- loss 用来描述模型当前预测和目标之间的差距。
- 优化器根据梯度更新模型参数。
- 实验观察需要结合曲线、数值和样本可视化。
- 调试时优先检查 shape、dtype、学习率和训练/评估模式。

这里的关键不是堆公式，而是把概念落实到可运行代码。

## 最小代码示例：小数据任务

这个代码块用于观察 `小数据任务`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

x, y = make_moons(n_samples=180, noise=0.25, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.45, random_state=42, stratify=y)
scaler = StandardScaler()
x_train = torch.tensor(scaler.fit_transform(x_train).astype(np.float32))
x_test = torch.tensor(scaler.transform(x_test).astype(np.float32))
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)
print("train size:", len(x_train), "test size:", len(x_test))

## 最小代码示例：训练函数

这个代码块用于观察 `训练函数`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
def run_model(use_dropout=False, weight_decay=0.0):
    layers = [nn.Linear(2, 64), nn.ReLU()]
    if use_dropout:
        layers.append(nn.Dropout(p=0.3))
    layers += [nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2)]
    model = nn.Sequential(*layers)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    train_accs, test_accs = [], []
    for _ in range(160):
        model.train()
        loss = loss_fn(model(x_train), y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            train_accs.append((model(x_train).argmax(1) == y_train).float().mean().item())
            test_accs.append((model(x_test).argmax(1) == y_test).float().mean().item())
    return train_accs, test_accs

plain_train, plain_test = run_model(False, 0.0)
reg_train, reg_test = run_model(True, 1e-3)

## 完整实验：对比曲线

这个代码块用于观察 `对比曲线`。运行后请重点看输出 shape、loss 或图像结果是否符合预期。

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(plain_train, label="plain train")
plt.plot(plain_test, label="plain test")
plt.plot(reg_train, label="regularized train", linestyle="--")
plt.plot(reg_test, label="regularized test", linestyle="--")
plt.title("Regularization comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 实验观察

正则化不一定让训练集准确率最高，但目标是改善泛化。观察训练集和测试集差距比只看训练 loss 更可靠。

从运行结果可以看到，代码输出不只是“能跑”，还可以帮助判断模型是否在按预期学习。建议把观察写进实验记录，而不是只保留最终准确率。

## 常见错误

- shape 不匹配：先打印输入、输出和标签 shape。
- dtype 不正确：分类标签通常是 `torch.long`，回归目标通常是 float。
- 忘记 `optimizer.zero_grad()`：梯度会累积。
- 学习率不合理：过大震荡，过小收敛慢。
- 评估阶段忘记 `model.eval()` 或 `torch.no_grad()`。

调试时可以先缩小数据集，确认模型能否在小样本上过拟合。

## 概念问答

**Q：为什么要从最小代码示例开始？**  
A：最小示例能隔离核心概念，降低同时排查多个问题的难度。

**Q：为什么每个实验都要看曲线或可视化？**  
A：单个最终数值不一定能解释训练过程，曲线和样本结果更容易暴露问题。

**Q：如果代码能运行但结果不好，先查什么？**  
A：优先检查数据、shape、label dtype、loss 使用方式和学习率。

## 延伸练习

- 调整 Dropout 概率。
- 只使用 weight decay 或只使用 Dropout 对比。

这些练习不需要一次完成。建议每次只改一个变量，并记录改动前后的结果。

## 小结

本节把一个深度学习概念落到了可运行的 PyTorch 实验中。继续学习下一节前，建议确认自己能解释：输入是什么、模型做了什么、loss 如何计算、结果是否符合预期。